In [1]:
import pandas as pd
import numpy as np
import pickle

In [2]:
books = pd.read_csv("CSV\Books.csv")

C:\Users\Aashish pathak\AppData\Local\Temp\ipykernel_17932\923019014.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv("CSV\Books.csv")


In [3]:
ratings = pd.read_csv("CSV\Ratings.csv")

In [4]:
users = pd.read_csv("CSV\\Users.csv")

In [5]:
print(books.shape)
print(ratings.shape)
print(users.shape)

(271360, 8)
(1149780, 3)
(278858, 3)


In [6]:
print(books.isnull().sum())
print(users.isnull().sum())
print(ratings.isnull().sum())

ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            3
dtype: int64
User-ID          0
Location         0
Age         110762
dtype: int64
User-ID        0
ISBN           0
Book-Rating    0
dtype: int64


In [7]:
print(books.duplicated().sum())
print(ratings.duplicated().sum())
print(users.duplicated().sum())

0
0
0


POPULARITY BASED RECOMMENDER SYSTEM

In [8]:
ratings_with_name = (ratings.merge(books, on = "ISBN"))
ratings_with_name
ratings_with_name.shape

(196829, 10)

In [9]:
num_rating_df = ratings_with_name.groupby("Book-Title").count()["Book-Rating"].reset_index()
num_rating_df.rename(columns={"Book-Rating":"num_ratings"}, inplace=True)
num_rating_df

,Book-Title,num_ratings
0,Earth Prayers From around the World: 365 Pray...,10
1,Final Fantasy Anthology: Official Strategy Gu...,4
2,Flight of Fancy: American Heiresses (Zebra Ba...,2
3,Little Comic Shop of Horrors (Give Yourself G...,4
4,Mystery Mile,2
...,...,...
68750,Ã?Â?lpiraten.,2
68751,Ã?Â?rger mit Produkt X. Roman.,4
68752,Ã?Â?sterlich leben.,1
68753,Ã?Â?stlich der Berge.,3


In [10]:
ratings_with_name["Book-Rating"] = pd.to_numeric(
    ratings_with_name["Book-Rating"], errors="coerce"
)

# Group by 'Book-Title' and calculate the mean of 'Book-Rating'
avg_rating_df = (
    ratings_with_name.groupby("Book-Title")["Book-Rating"].mean().reset_index()
)

# Rename the column to 'avg_rating'
avg_rating_df.rename(columns={"Book-Rating": "avg_rating"}, inplace=True)
avg_rating_df

,Book-Title,avg_rating
0,Earth Prayers From around the World: 365 Pray...,5.000000
1,Final Fantasy Anthology: Official Strategy Gu...,5.000000
2,Flight of Fancy: American Heiresses (Zebra Ba...,4.000000
3,Little Comic Shop of Horrors (Give Yourself G...,1.250000
4,Mystery Mile,0.000000
...,...,...
68750,Ã?Â?lpiraten.,0.000000
68751,Ã?Â?rger mit Produkt X. Roman.,5.250000
68752,Ã?Â?sterlich leben.,7.000000
68753,Ã?Â?stlich der Berge.,2.666667


In [11]:
popular_df = num_rating_df.merge(avg_rating_df, on = "Book-Title")
popular_df

,Book-Title,num_ratings,avg_rating
0,Earth Prayers From around the World: 365 Pray...,10,5.000000
1,Final Fantasy Anthology: Official Strategy Gu...,4,5.000000
2,Flight of Fancy: American Heiresses (Zebra Ba...,2,4.000000
3,Little Comic Shop of Horrors (Give Yourself G...,4,1.250000
4,Mystery Mile,2,0.000000
...,...,...,...
68750,Ã?Â?lpiraten.,2,0.000000
68751,Ã?Â?rger mit Produkt X. Roman.,4,5.250000
68752,Ã?Â?sterlich leben.,1,7.000000
68753,Ã?Â?stlich der Berge.,3,2.666667


In [12]:
popular_df = popular_df[popular_df["num_ratings"]>=50].sort_values("avg_rating",ascending=False)

In [13]:
popular_df.shape

(262, 3)

In [14]:
books.columns

Index(['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher',
       'Image-URL-S', 'Image-URL-M', 'Image-URL-L'],
      dtype='object')

In [15]:
popular_df = popular_df.merge(books, on="Book-Title").drop_duplicates("Book-Title")[
    ["Book-Title", "Book-Author", "Image-URL-M", "num_ratings", "avg_rating"]
]

Collaborative Filtering Based Recommender System

In [16]:
x = ratings_with_name.groupby("User-ID").count()["Book-Rating"]>100
educated_users = x[x].index
educated_users.shape

(222,)

In [17]:
filtered_ratings = ratings_with_name[ratings_with_name["User-ID"].isin(educated_users)]
filtered_ratings.shape

(46693, 10)

In [18]:
y = filtered_ratings.groupby("Book-Title").count()["Book-Rating"]>=20
famous_books = y[y].index
famous_books.shape

(101,)

In [19]:
final_ratings = filtered_ratings[filtered_ratings["Book-Title"].isin(famous_books)]

In [20]:
pivot_tble = final_ratings.pivot_table(index = 'Book-Title', columns = 'User-ID', values = "Book-Rating")
pivot_tble.shape

(101, 185)

In [21]:
pivot_tble.fillna(0,inplace=True)
pivot_tble

User-ID,3363,6251,6575,7346,11601,11676,12538,12824,15408,16634,...,261105,262998,265115,266226,269566,271284,274061,274308,275970,278418
Book-Title,,,,,,,,,,,,,,,,,,,,,
16 Lighthouse Road,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
204 Rosewood Lane,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A 2nd Helping of Chicken Soup for the Soul (Chicken Soup for the Soul Series (Paper)),0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"A Child Called \It\"": One Child's Courage to Survive""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A Fine Balance,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Turtle Moon,0.0,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0
Up Island: A Novel,0.0,0.0,0.0,0.0,0.0,9.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"Welcome to the World, Baby Girl!",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [22]:
from sklearn.metrics.pairwise import cosine_similarity
similarity_scores = cosine_similarity(pivot_tble)
similarity_scores
similarity_scores.shape


(101, 101)

In [23]:
def recommend(book_name):
    # Check if the book_name exists in the index
    if book_name not in pivot_tble.index:
        print("Book not found in index.")
        return None

    # Index fetch
    index = np.where(pivot_tble.index == book_name)[0]

    # Check if index is empty
    if len(index) == 0:
        print("Index is empty.")
        return None

    index = index[0]

    # Continue with your remaining code
    similar_items = sorted(
        list(enumerate(similarity_scores[index])), key=lambda x: x[1], reverse=True
    )[1:6]

    data = []
    for i in similar_items:
        item = []
        temp_df = books[books["Book-Title"] == pivot_tble.index[i[0]]]
        item.extend(list(temp_df.drop_duplicates("Book-Title")["Book-Title"].values))
        item.extend(list(temp_df.drop_duplicates("Book-Title")["Book-Author"].values))
        item.extend(list(temp_df.drop_duplicates("Book-Title")["Image-URL-M"].values))

        data.append(item)

    return data

In [24]:
recommend("Turtle Moon")

[['Heartbreaker',
  'Julie Garwood',
  'https://images.amazon.com/images/P/0671034006.01.MZZZZZZZ.jpg'],
 ['The Bridges of Madison County',
  'Robert James Waller',
  'https://images.amazon.com/images/P/044651652X.01.MZZZZZZZ.jpg'],
 ['Red Storm Rising',
  'Tom Clancy',
  'https://images.amazon.com/images/P/042510107X.01.MZZZZZZZ.jpg'],
 ['Snow Falling on Cedars',
  'David Guterson',
  'https://images.amazon.com/images/P/0151001006.01.MZZZZZZZ.jpg'],
 ['STONES FROM THE RIVER',
  'Ursula Hegi',
  'https://images.amazon.com/images/P/0684844729.01.MZZZZZZZ.jpg']]

In [25]:
pivot_tble.index[100]

"What to Expect When You're Expecting (Revised Edition)"

In [26]:
books.drop_duplicates('Book-Title')

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,https://images.amazon.com/images/P/0195153448....,https://images.amazon.com/images/P/0195153448....,https://images.amazon.com/images/P/0195153448....
1,2005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,https://images.amazon.com/images/P/0002005018....,https://images.amazon.com/images/P/0002005018....,https://images.amazon.com/images/P/0002005018....
2,60973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,https://images.amazon.com/images/P/0060973129....,https://images.amazon.com/images/P/0060973129....,https://images.amazon.com/images/P/0060973129....
3,374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,https://images.amazon.com/images/P/0374157065....,https://images.amazon.com/images/P/0374157065....,https://images.amazon.com/images/P/0374157065....
4,393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,https://images.amazon.com/images/P/0393045218....,https://images.amazon.com/images/P/0393045218....,https://images.amazon.com/images/P/0393045218....
...,...,...,...,...,...,...,...,...
271354,449906736,Flashpoints: Promise and Peril in a New World,Robin Wright,1993,Ballantine Books,https://images.amazon.com/images/P/0449906736....,https://images.amazon.com/images/P/0449906736....,https://images.amazon.com/images/P/0449906736....
271356,525447644,From One to One Hundred,Teri Sloat,1991,Dutton Books,https://images.amazon.com/images/P/0525447644....,https://images.amazon.com/images/P/0525447644....,https://images.amazon.com/images/P/0525447644....
271357,006008667X,Lily Dale : The True Story of the Town that Ta...,Christine Wicker,2004,HarperSanFrancisco,https://images.amazon.com/images/P/006008667X....,https://images.amazon.com/images/P/006008667X....,https://images.amazon.com/images/P/006008667X....
271358,192126040,Republic (World's Classics),Plato,1996,Oxford University Press,https://images.amazon.com/images/P/0192126040....,https://images.amazon.com/images/P/0192126040....,https://images.amazon.com/images/P/0192126040....


In [27]:
pickle.dump(popular_df, open("popular.pkl", "wb"))
pickle.dump(pivot_tble, open("pt.pkl", "wb"))
pickle.dump(books, open("books.pkl", "wb"))
pickle.dump(similarity_scores, open("similarity_scores.pkl", "wb"))